# ECNet - Dataset Inventory

Counts **exactly how many real and AI-generated videos** came from each attached
Kaggle dataset, with the source and generator breakdown underneath.

Per dataset it reports:
- total unique videos, split real vs AI-generated
- the upstream source of each (Pexels, YouTube UGC, AVGen-Bench, ...)
- which generator produced each AI video (`avg_<model>_<hash>`)
- how much overlaps between datasets, so the totals are not double-counted

**Setup:** Add Input -> attach `hybridframeextraction`, `frames-joman-10000`,
`extracted-frames-10000`, `extract-frames-gdrive` -> Run All.

Read-only; nothing is modified. Results are written to `/kaggle/working/`.

In [ ]:
# ========================= 1. LOCATE THE DATASETS ============================
import os, re, json, collections
from pathlib import Path
import pandas as pd

WANTED = ["hybridframeextraction", "frames-joman-10000",
          "extracted-frames-10000", "extract-frames-gdrive"]

def resolve(slug):
    """Kaggle mounts vary: /kaggle/input/<slug> or /kaggle/input/datasets/<user>/<slug>."""
    direct = Path("/kaggle/input") / slug
    if direct.is_dir():
        return direct
    for base in Path("/kaggle/input").glob("datasets/*/" + slug):
        if base.is_dir():
            return base
    for base in Path("/kaggle/input").rglob(slug):
        if base.is_dir():
            return base
    return None

DATASETS = {}
for slug in WANTED:
    p = resolve(slug)
    DATASETS[slug] = p
    print(f"  {slug:<28} " + (f"OK   {p}" if p else "NOT ATTACHED"))

missing = [s for s, p in DATASETS.items() if p is None]
if missing:
    print(f"\n!! Not attached: {missing}")
    print("   Add Input -> Datasets, or the counts below are partial.")

known = {str(p) for p in DATASETS.values() if p}
extra = [d for d in Path("/kaggle/input").iterdir()
         if d.is_dir() and str(d) not in known and d.name != "datasets"]
if extra:
    print("\nAlso attached (not in the requested list):")
    for d in extra:
        print("   ", d.name)

In [ ]:
# ========================= 2. LOAD EVERY INDEX CSV ===========================
# The extraction stage writes an index CSV per run. That is the authoritative
# record, and far faster than walking millions of frame files.

def find_csvs(root, limit=60):
    out = []
    for dirpath, _d, files in os.walk(root):
        for fn in files:
            if fn.lower().endswith(".csv"):
                out.append(Path(dirpath) / fn)
                if len(out) >= limit:
                    return out
    return out

def pick(cols, *keywords):
    """First column whose name contains any keyword."""
    for c in cols:
        lc = str(c).lower()
        if any(k in lc for k in keywords):
            return c
    return None

frames = []
for slug, root in DATASETS.items():
    if root is None:
        continue
    csvs = find_csvs(root)
    print(f"\n{slug}: {len(csvs)} CSV(s)")
    for p in csvs:
        try:
            df = pd.read_csv(p, low_memory=False)
        except Exception as e:
            print(f"   ! {p.name}: {type(e).__name__}")
            continue
        if df.empty:
            continue
        c_vid   = pick(df.columns, "video_id", "video", "stem", "clip")
        c_path  = pick(df.columns, "frame_path", "path", "file", "image")
        c_src   = pick(df.columns, "source", "generator", "platform", "dataset")
        c_label = pick(df.columns, "label", "class")
        print(f"   {p.name:<32} rows={len(df):>9,}  vid={c_vid} src={c_src} label={c_label}")
        frames.append(pd.DataFrame({
            "dataset":   slug,
            "csv":       p.name,
            "video_id":  df[c_vid].astype(str) if c_vid else (
                         df[c_path].astype(str) if c_path else ""),
            "raw_src":   df[c_src].astype(str) if c_src else "",
            "raw_label": df[c_label].astype(str) if c_label is not None else "",
            "path":      df[c_path].astype(str) if c_path else "",
        }))

covered = {f["dataset"].iloc[0] for f in frames} if frames else set()
print(f"\nDatasets with an index CSV: {sorted(covered) or 'none'}")

In [ ]:
# ========================= 3. FALLBACK: WALK CSV-LESS DATASETS ===============
# A dataset shipping no index CSV would otherwise be missing from every total
# below, silently undercounting the corpus. Walk its filenames instead and
# append the rows, so one pipeline feeds all the tables.
MEDIA = {".jpg", ".jpeg", ".png", ".webp", ".bmp",
         ".mp4", ".mov", ".mkv", ".webm", ".avi", ".m4v"}
CAP = 600_000

todo = [(s, p) for s, p in DATASETS.items() if p and s not in covered]
if not todo:
    print("Every attached dataset had an index CSV -- nothing to walk.")
for slug, root in todo:
    rows, n, capped = [], 0, False
    for dirpath, _d, files in os.walk(root):
        rel = os.path.relpath(dirpath, root)
        for fn in files:
            if Path(fn).suffix.lower() not in MEDIA:
                continue
            rows.append(os.path.join(rel, fn) if rel != "." else fn)
            n += 1
            if n >= CAP:
                capped = True
                break
        if capped:
            break
    print(f"  {slug}: walked {n:,} media files{'  (CAPPED)' if capped else ''}")
    if rows:
        frames.append(pd.DataFrame({
            "dataset": slug, "csv": "(filesystem)",
            "video_id": rows, "raw_src": "", "raw_label": "", "path": rows,
        }))

FR = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
print(f"\nTotal frame rows: {len(FR):,}")
if FR.empty:
    print("Nothing found -- check that the datasets are attached.")

In [ ]:
# ========================= 3. NORMALISE VIDEO ID + CLASS =====================
PREFIX = [("ugc_", "youtube_ugc"), ("vis_", "vision_devices"),
          ("pexl_", "pexels"), ("pex_", "pexels"), ("dact_", "deepaction"),
          ("avg_", "avgen_bench"), ("gvb_", "genvidbench")]

# AVGen-Bench, GenVidBench and DeepAction are generated-video corpora;
# the rest are camera footage.
AI_SOURCES   = {"avgen_bench", "genvidbench", "deepaction"}
REAL_SOURCES = {"youtube_ugc", "vision_devices", "pexels"}

def strip_frame_id(s):
    """'<stem>_f0001.jpg' -> '<stem>'"""
    s = Path(str(s)).stem
    if "_f" in s and s.rsplit("_f", 1)[-1].isdigit():
        s = s.rsplit("_f", 1)[0]
    return s

def source_of(stem):
    for pre, tag in PREFIX:
        if stem.startswith(pre):
            return tag
    return "untagged"

def class_of(row):
    """Class from the strongest available evidence, in order of trust."""
    lab = str(row["raw_label"]).strip().lower()
    if lab in ("1", "1.0", "ai", "ai_generated", "fake", "generated"):
        return "ai"
    if lab in ("0", "0.0", "real", "authentic"):
        return "real"
    src = str(row["raw_src"]).strip().lower()
    if src in ("real", "authentic"):
        return "real"
    if src in ("ai", "ai_generated", "fake", "generated"):
        return "ai"
    # match whole path SEGMENTS: a substring test would call any path
    # containing "real" real, including dataset names that merely contain it
    parts = {p for p in re.split(r"[\\/]", str(row["path"]).lower()) if p}
    if parts & {"ai_generated", "ai", "fake", "generated"}:
        return "ai"
    if parts & {"real", "authentic"}:
        return "real"
    s = row["_source"]
    if s in AI_SOURCES:
        return "ai"
    if s in REAL_SOURCES:
        return "real"
    return "unknown"

if not FR.empty:
    FR["stem"] = FR["video_id"].map(strip_frame_id)
    FR["_source"] = FR["stem"].map(source_of)
    FR["cls"] = FR.apply(class_of, axis=1)

    VID = FR.drop_duplicates(["dataset", "stem"])[
        ["dataset", "stem", "_source", "cls"]].reset_index(drop=True)

    print(f"unique (dataset, video) pairs : {len(VID):,}")
    print(f"unique videos overall         : {VID['stem'].nunique():,}")
    print()
    print(FR["cls"].value_counts().to_string())

    unk = int((VID["cls"] == "unknown").sum())
    if unk:
        print(f"\n!! {unk:,} videos unclassified -- sample stems:")
        for s in VID[VID["cls"] == "unknown"]["stem"].head(8):
            print("     ", s)

## The answer - real vs AI per dataset

In [ ]:
# ========================= 4. REAL vs AI, PER DATASET ========================
if not FR.empty:
    vids = VID.pivot_table(index="dataset", columns="cls", aggfunc="size", fill_value=0)
    frms = FR.pivot_table(index="dataset", columns="cls", aggfunc="size", fill_value=0)
    for t in (vids, frms):
        for c in ("real", "ai", "unknown"):
            if c not in t.columns:
                t[c] = 0

    print("VIDEOS (unique within each dataset)")
    print(f"{'dataset':<28}{'real':>10}{'ai':>10}{'unknown':>10}{'total':>10}")
    print("-" * 68)
    for ds in vids.index:
        r = int(vids.loc[ds, "real"]); a = int(vids.loc[ds, "ai"]); u = int(vids.loc[ds, "unknown"])
        print(f"{ds:<28}{r:>10,}{a:>10,}{u:>10,}{r + a + u:>10,}")
    tr = int(vids["real"].sum()); ta = int(vids["ai"].sum()); tu = int(vids["unknown"].sum())
    print("-" * 68)
    print(f"{'TOTAL (overlap included)':<28}{tr:>10,}{ta:>10,}{tu:>10,}{tr + ta + tu:>10,}")

    print("\n\nFRAMES")
    print(f"{'dataset':<28}{'real':>12}{'ai':>12}{'unknown':>10}{'total':>12}")
    print("-" * 74)
    for ds in frms.index:
        r = int(frms.loc[ds, "real"]); a = int(frms.loc[ds, "ai"]); u = int(frms.loc[ds, "unknown"])
        print(f"{ds:<28}{r:>12,}{a:>12,}{u:>10,}{r + a + u:>12,}")
    print("-" * 74)
    print(f"{'TOTAL':<28}{int(frms['real'].sum()):>12,}{int(frms['ai'].sum()):>12,}"
          f"{int(frms['unknown'].sum()):>10,}{len(FR):>12,}")

In [ ]:
# ========================= 5. UPSTREAM SOURCE BREAKDOWN ======================
if not FR.empty:
    for cls in ("real", "ai"):
        sub = VID[VID["cls"] == cls]
        if sub.empty:
            continue
        print(f"\n=== {cls.upper()} -- {len(sub):,} videos by upstream source ===")
        t = sub.pivot_table(index="_source", columns="dataset", aggfunc="size", fill_value=0)
        t["TOTAL"] = t.sum(axis=1)
        print(t.sort_values("TOTAL", ascending=False).to_string())

In [ ]:
# ========================= 6. GENERATORS (AI ONLY) ===========================
# AVGen-Bench encodes the model as avg_<model>_<hash>.
gens = collections.Counter()
if not FR.empty:
    ai = VID[VID["cls"] == "ai"]
    for stem in ai["stem"]:
        if stem.startswith("avg_"):
            tail = stem[4:]
            gens[tail.rsplit("_", 1)[0] if "_" in tail else tail] += 1

    if gens:
        tot = sum(gens.values())
        print(f"{len(gens)} distinct generators, {tot:,} tagged videos\n")
        print(f"{'generator':<44}{'videos':>9}{'share':>9}")
        print("-" * 62)
        for g, c in gens.most_common():
            print(f"{g:<44}{c:>9,}{100 * c / tot:>8.1f}%")
        untagged = len(ai) - tot
        if untagged:
            print("-" * 62)
            print(f"{'UNTAGGED (legacy/stock pool)':<44}{untagged:>9,}"
                  f"{100 * untagged / max(len(ai), 1):>8.1f}%")
    else:
        print("No avg_-tagged videos found.")

In [ ]:
# ========================= 7. OVERLAP BETWEEN DATASETS =======================
# The same video can sit in more than one dataset, so summing datasets
# overstates the corpus. This is the honest unique figure.
if not FR.empty:
    per_ds = {ds: set(g["stem"]) for ds, g in VID.groupby("dataset")}
    names = list(per_ds)

    print("Shared videos between datasets")
    print(f"{'':<28}" + "".join(f"{n[:16]:>18}" for n in names))
    for a in names:
        row = f"{a:<28}"
        for b in names:
            row += f"{len(per_ds[a] & per_ds[b]):>18,}"
        print(row)

    allv = set().union(*per_ds.values()) if per_ds else set()
    summed = sum(len(v) for v in per_ds.values())
    print(f"\nSum across datasets : {summed:,}")
    print(f"Unique videos       : {len(allv):,}")
    print(f"Counted twice+      : {summed - len(allv):,}")

    print("\nUNIQUE CORPUS (deduplicated)")
    print(VID.drop_duplicates("stem")["cls"].value_counts().to_string())

In [ ]:
# ========================= 8. SAVE FOR THE PAPER =============================
OUT = Path("/kaggle/working")
if not FR.empty:
    VID.to_csv(OUT / "dataset_videos.csv", index=False)
    (VID.pivot_table(index=["dataset", "_source"], columns="cls",
                     aggfunc="size", fill_value=0)
        .reset_index().to_csv(OUT / "dataset_summary.csv", index=False))

    uniq = VID.drop_duplicates("stem")
    totals = {
        "videos_unique":  int(len(uniq)),
        "videos_real":    int((uniq["cls"] == "real").sum()),
        "videos_ai":      int((uniq["cls"] == "ai").sum()),
        "videos_unknown": int((uniq["cls"] == "unknown").sum()),
        "frames_total":   int(len(FR)),
        "datasets":       {k: (str(v) if v else None) for k, v in DATASETS.items()},
        "generators":     dict(gens),
    }
    (OUT / "dataset_totals.json").write_text(json.dumps(totals, indent=2))
    print(json.dumps(totals, indent=2)[:1400])
    print("\nWrote dataset_videos.csv, dataset_summary.csv, dataset_totals.json")
    print("Save Version -> they appear in the Output tab.")

## Reading the results

**Cell 4 is the headline** - real vs AI per dataset. A video present in two
datasets is counted once in each there; **cell 7** gives the deduplicated corpus
figure, and that is the number to quote in the paper.

**`unknown` should be near zero.** A large count means videos whose class could
not be established from the CSV label, the directory, or the filename prefix.
Inspect the sample stems printed by cell 3 and extend `PREFIX` or `class_of`.

**Compare against training.** ECNet-7 trained on 11,500 per class after
`cap_per_class: 11500`, and its checkpoint records 3,453 validation videos. If
the totals here are larger, the cap discarded the surplus - and *which* videos
it kept determines whether the generators stayed balanced. That difference is
worth reporting.